# NEET analysis
This notebook loads the cleaned CPS ASEC data from Google Drive and takes a first look. Run it top to bottom.

## Setup
Import the tools and set the file path in one place.

In [ ]:
import pandas as pd  # tables of data
import numpy as np  # math, used for weighted averages

data_path = "/content/drive/My Drive/neet_data.csv"  # where the cleaned data lives on Drive

## Mount Google Drive
This lets Colab read files from your Drive. Colab will ask you to allow access.

In [ ]:
from google.colab import drive  # Colab's Google Drive tool
drive.mount("/content/drive")  # connect Google Drive

## Load the data

In [ ]:
df = pd.read_csv(data_path)  # load the cleaned data into a table

## Shape and column names
Shape is (number of people, number of variables).

In [ ]:
print(df.shape)  # rows (people) and columns (variables)
print(list(df.columns))  # names of all columns

## Summary table
All numbers use the person weight `asecwt`. N is the number of people with a value (not weighted). Mean, SD, Min and Max are weighted where it applies. Means of 0/1 variables are shares (0.5 = 50%).

In [ ]:
rows = {}  # empty dictionary, one row per variable
for var in df.select_dtypes("number").columns:  # loop over the number columns
    if var == "asecwt":  # the weight itself is not summarized
        continue  # skip it
    d = df.dropna(subset=[var])  # keep people with a value for this variable
    x = d[var]  # the values
    w = d["asecwt"]  # the person weights
    mean = np.average(x, weights=w)  # weighted mean
    sd = np.sqrt(np.average((x - mean) ** 2, weights=w))  # weighted standard deviation
    rows[var] = {"N": len(d), "Mean": mean, "SD": sd, "Min": x.min(), "Max": x.max()}  # one row of the table

summary = pd.DataFrame(rows).T  # dictionary -> table, one row per variable
summary["N"] = summary["N"].astype(int)  # N is a whole number
print(summary.to_string(float_format="{:.1f}".format))  # print with 1 decimal

## NEET rate by year, age group and sex
The NEET rate is the weighted percent of people with `neet = 1`, among people whose `neet` is known. People with a missing `neet` are left out (they are not counted as 0). Rows are survey years. Columns are age group first, then sex (Men, then Women). Weights use `asecwt`.

In [ ]:
def wmean(data, var):  # weighted mean of one variable
    d = data.dropna(subset=[var])  # leave out people with a missing value
    return np.average(d[var], weights=d["asecwt"])  # weighted average using asecwt

if "agegrp" not in df.columns:  # make age groups only if the file does not have them
    df["agegrp"] = pd.cut(df["age"], bins=[16, 20, 25, 30], labels=["16-19", "20-24", "25-29"], right=False)  # 16-19, 20-24, 25-29

df["sex"] = pd.Categorical(df["female"].map({0: "Men", 1: "Women"}), categories=["Men", "Women"], ordered=True)  # words for female, Men first

known = df.dropna(subset=["neet"])  # keep only people whose neet is known
keys = ["year", "agegrp", "sex"]  # group by year, age group, sex

rate = known.groupby(keys, observed=True).apply(wmean, "neet", include_groups=False) * 100  # weighted NEET rate in percent
rate = rate.unstack(["agegrp", "sex"])  # years down the rows, age group and sex across
people = known.groupby(keys, observed=True).size().unstack(["agegrp", "sex"]).astype(float)  # unweighted respondents, same layout

rate.loc[rate.index < 2013, "25-29"] = np.nan  # ages 25-29 before 2013: blank, not 0
people.loc[people.index < 2013, "25-29"] = np.nan  # same blanks in the counts

print("NEET rate, % of people (weighted with asecwt). Rows: survey year.")  # table title
print(rate.to_string(float_format="{:.1f}".format, na_rep=""))  # rate table, 1 decimal, blanks stay blank
print()  # empty line
print("Unweighted number of respondents in each cell (neet not missing).")  # counts title
print(people.to_string(float_format="{:.0f}".format, na_rep=""))  # counts table, whole numbers, blanks stay blank
print()  # empty line
print("Ages 25-29 are blank before 2013 because school enrollment (SCHLCOLL) was not asked of that age group.")  # required note

**What one cell means.** Take the cell in the row for one survey year and the column for one age group and sex (for example, the row 2019 and the column 20-24, Women). Among women aged 20-24 in the survey of that year whose NEET status is known, it is the percent who were NEET: not employed and not in school. People are counted by how many people in the country they represent (`asecwt`), so the number describes the population, not just the people surveyed. A cell of 10.0 means 10 out of every 100 such people were NEET. The count table below it shows how many survey respondents sit behind each cell: a cell with few respondents is less reliable. A blank cell means the rate is not available, not zero.